# Phase 1 — 40M Mini-TradeFM pretrain on Colab Pro+

Trains the 40M TradeFM on tokenized 0DTE microstructure shards. Designed for:

  - Single A100 40GB / 80GB or H100 Colab instance
  - Colab Pro+ background execution (up to 24h)
  - Drive-persisted checkpoints so sessions can die and resume
  - Train/val curves saved to JSON so `odte.train.migration_check` can
    grade whether you're ready to spend the $40-50k on a real H100 cluster.

**Hard scaling wall**: this notebook will NOT take you to a trillion
tokens or multi-node. When the migration gate says GO, move to
`infra/gcp/phase2_a3mega.sh` for the full 524M run.

**Before running**: *Runtime → Change runtime type → A100* (or H100 if available).

## 1. GPU check + Pro+ background-exec note

In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available())
print('Pro+ BG-exec: close browser safely — runtime persists up to 24h.')

name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB
torch: 2.10.0+cu128 cuda: True
Pro+ BG-exec: close browser safely — runtime persists up to 24h.


## 2. Mount Drive — this IS the checkpoint store

In [2]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/tradefm_ckpts'
!mkdir -p {DRIVE_ROOT}
!ls -la {DRIVE_ROOT}

Mounted at /content/drive
total 210
-rw------- 1 root root 206389 Apr 17 23:27 dml_pricer.pt
drwx------ 2 root root   4096 Apr 17 23:32 phase1_shards
drwx------ 2 root root   4096 Apr 17 23:32 tradefm_40m


## 3. Clone private repo + install (uses a Colab Secret)

**One-time setup** before running the next cell:
1. On GitHub: *Settings → Developer settings → Personal access tokens → Tokens (classic) → Generate new token (classic)*.
   Scope = `repo`. Expiry 30 days is fine.
2. In Colab: left-side **🔑 key icon** → *+ Add new secret*. Name: **`GITHUB_TOKEN`**. Value: the PAT. Toggle *Notebook access* ON for this notebook.

If the token ever leaks (e.g. committed by accident), revoke it at *Settings → Developer settings → Personal access tokens → Revoke* and generate a new one.

In [3]:
import os, getpass
from google.colab import userdata

GH_USER = 'nahomar'
REPO_NAME = 'market-pattern-bot'

# Try Colab Secret first. Falls back to a one-time paste if it times out.
# TimeoutException usually = secret set but "Notebook access" toggle OFF.
TOKEN = None
try:
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception as e:
    print(f'Colab Secret unavailable ({type(e).__name__}). '
          'Toggle "Notebook access" ON in the 🔑 panel, or paste PAT below.')

if not TOKEN:
    TOKEN = getpass.getpass('GitHub PAT (ghp_...): ').strip()
if not TOKEN:
    raise RuntimeError('No token provided.')

AUTH_URL  = f'https://{TOKEN}@github.com/{GH_USER}/{REPO_NAME}.git'
CLEAN_URL = f'https://github.com/{GH_USER}/{REPO_NAME}.git'
CLONE_DIR = f'/content/{REPO_NAME}'

if not os.path.exists(CLONE_DIR):
    !git clone --depth=1 $AUTH_URL $CLONE_DIR
os.chdir(CLONE_DIR)
!git remote set-url origin $CLEAN_URL
del TOKEN, AUTH_URL

!pip install -q -r requirements.txt
!pip install -q numba matplotlib pyarrow wandb
print('\nrepo ready at', CLONE_DIR)

Cloning into '/content/market-pattern-bot'...
remote: Enumerating objects: 151, done.
remote: Counting objects: 100% (151/151), done.
remote: Compressing objects: 100% (150/150), done.
remote: Total 151 (delta 0), reused 139 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (151/151), 218.24 KiB | 21.82 MiB/s, done.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 kB 20.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 17.3 MB/s eta 0:00:00

repo ready at /content/market-pattern-bot


## 4. Data: synth digital twin + shard pack

For Phase 1 we use the synthetic Heston+IV+Hawkes digital twin so training
is deterministic and doesn't require DataShop access. The same code path
swaps in real OPRA shards on the H100 cluster later.

In [ ]:
import sys, shutil
sys.path.insert(0, '/content/market-pattern-bot')

from pathlib import Path
from odte.synth_options import SessionSpec, generate_session
from odte.dml_pricer import bs_price_call
from odte.data import DataShopPacker
import pandas as pd
import numpy as np
import torch

# ---------------------------------------------------------------------------
# Diverse multi-strike synth — prevents the 40M transformer from memorizing.
# Previous cell wrote a trivial single-strike CSV; model overfit to 100%.
# Now: 12 sessions across 3 IV regimes, 21 strikes × {C,P}, BS-priced
# bids/asks with IV- and moneyness-scaled spreads.
# ---------------------------------------------------------------------------

CSV_DIR = Path('/content/fake_datashop'); shutil.rmtree(CSV_DIR, ignore_errors=True)
CSV_DIR.mkdir(parents=True, exist_ok=True)

REGIMES = [
    ('calm',     0.10,  0.00), ('calm',     0.10,  0.03), ('calm',     0.10, -0.03),
    ('normal',   0.20,  0.00), ('normal',   0.20,  0.05), ('normal',   0.20, -0.05),
    ('normal',   0.20,  0.02), ('elevated', 0.35,  0.00), ('elevated', 0.35,  0.08),
    ('elevated', 0.35, -0.08), ('elevated', 0.35,  0.12), ('elevated', 0.35, -0.12),
]

for d, (regime, base_iv, mu) in enumerate(REGIMES):
    spec = SessionSpec(S0=5500.0 + 50.0 * (d - 6),
                       n_steps=3600, dt_seconds=6.0, seed=100 + d)
    under, trades, chain = generate_session(spec, write=False)

    chain = chain.copy()
    iv_scale = base_iv / max(chain['iv'].mean(), 1e-6)
    chain['iv'] = (chain['iv'] * iv_scale).clip(0.03, 1.5)

    S_t = torch.tensor(chain['S'].values, dtype=torch.float32, device='cuda')
    K_t = torch.tensor(chain['strike'].values, dtype=torch.float32, device='cuda')
    tau_t = torch.tensor(chain['tau_years'].values, dtype=torch.float32, device='cuda')
    sig_t = torch.tensor(chain['iv'].values, dtype=torch.float32, device='cuda')
    r_t = torch.zeros_like(S_t)
    price_c = bs_price_call(S_t, K_t, tau_t, sig_t, r_t).cpu().numpy()
    price_p = (price_c - S_t.cpu().numpy() + K_t.cpu().numpy()).clip(min=0.01)
    chain['theo'] = np.where(chain['cp'] == 'C', price_c, price_p)

    moneyness = (chain['strike'] / chain['S'] - 1).abs()
    spread_bps = (chain['iv'] * (1 + 5 * moneyness) * 0.005 + 0.002)
    half = (chain['theo'] * spread_bps / 2).clip(lower=0.05)
    chain['bid'] = (chain['theo'] - half).clip(lower=0.05)
    chain['ask'] = (chain['theo'] + half).clip(lower=chain['bid'] + 0.05)

    df = pd.DataFrame({
        'underlying_symbol': 'SPX',
        'quote_datetime': pd.Timestamp(f'2026-04-{17+d:02d} 09:30') + pd.to_timedelta(chain['ts_sec'], unit='s'),
        'root': 'SPX', 'expiration': f'2026-04-{17+d:02d}',
        'strike': chain['strike'], 'option_type': chain['cp'],
        'bid': chain['bid'],  'bid_size': np.random.default_rng(d).integers(5, 80, len(chain)),
        'ask': chain['ask'],  'ask_size': np.random.default_rng(d+1).integers(5, 80, len(chain)),
        'trade_volume': 0, 'implied_volatility': chain['iv'],
    })
    df.to_csv(CSV_DIR / f'day_{d:02d}_{regime}.csv', index=False)
    print(f'  day {d} {regime:8s} S0={spec.S0:.0f} iv={base_iv:.2f} mu={mu:+.2f} rows={len(df)}')

print(f'\nwrote {len(REGIMES)} diverse synth-day CSVs')

# Small shards so held-out eval has shards to reserve
SHARD_DIR = Path(f'{DRIVE_ROOT}/phase1_shards'); shutil.rmtree(SHARD_DIR, ignore_errors=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
packer = DataShopPacker(out_dir=SHARD_DIR, n_buckets=64, shard_rows=8_000)
packer.fit_tokenizer(sorted(CSV_DIR.glob('*.csv')))
shards = packer.pack(sorted(CSV_DIR.glob('*.csv')))
print(f'packed {len(shards)} shards → {SHARD_DIR}')
assert len(shards) >= 3, f'need >=3 shards for held-out eval, got {len(shards)}'

## 5. Train — write loss history to Drive every N steps

In [8]:
import os, gc, sys, importlib, subprocess
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Verify repo is on the latest commit
head = subprocess.check_output(
    ['git','-C','/content/market-pattern-bot','rev-parse','--short','HEAD']
).decode().strip()
print(f'[diag] repo HEAD = {head}  (must be d7a7e64 or later)')

# Nuke stale GPU + Python state
import torch
for name in ('model','opt','stats','cfg','final_cfg'):
    globals().pop(name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
free, total = torch.cuda.mem_get_info()
print(f'[diag] GPU free after cleanup: {free/1024**3:.1f} / {total/1024**3:.1f} GB')

# Reload the patched modules (guards against stale class definitions)
for mod in ('odte.transformer_tradefm','odte.train.pretrain_tradefm'):
    if mod in sys.modules:
        importlib.reload(sys.modules[mod])

from odte.train.pretrain_tradefm import TrainArgs, pretrain, load_config
from pathlib import Path
import yaml

ckpt_dir = f'{DRIVE_ROOT}/tradefm_40m'
Path(ckpt_dir).mkdir(parents=True, exist_ok=True)

mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
CFG_40M = '/content/market-pattern-bot/configs/tradefm_40m.yml'
CFG_SMK = '/content/market-pattern-bot/configs/tradefm_smoke.yml'

def _patch_cfg(path, **overrides):
    with open(path) as f: cfg = yaml.safe_load(f)
    cfg.update(overrides)
    with open(path,'w') as f: yaml.safe_dump(cfg, f)
    return cfg

if mem_gb >= 70:
    cfg_path, batch, accum = CFG_40M, 16, 2
    _patch_cfg(cfg_path, grad_checkpointing=False)
elif mem_gb >= 35:
    # A100 40G — aggressive: ctx=1024, batch=1, checkpointing + bf16
    cfg_path, batch, accum = CFG_40M, 1, 32
    _patch_cfg(cfg_path, grad_checkpointing=True, ctx_len=1024)
else:
    cfg_path, batch, accum = CFG_SMK, 8, 1

final_cfg = load_config(cfg_path)
print(f'[diag] GPU mem={mem_gb:.1f}GB → config={Path(cfg_path).name}')
print(f'       d_model={final_cfg.d_model}  n_layers={final_cfg.n_layers}  '
      f'ctx_len={final_cfg.ctx_len}  vocab={final_cfg.vocab}')
print(f'       batch={batch} grad_accum={accum} '
      f'grad_checkpointing={final_cfg.grad_checkpointing}')

stats = pretrain(TrainArgs(
    shard_glob=str(SHARD_DIR / 'opra_*.parquet'),
    ckpt_dir=ckpt_dir,
    config_path=cfg_path,
    steps=5000, batch=batch, grad_accum=accum,
    ckpt_every=500, log_every=50,
    device='cuda', seed=0, max_shards=None,
))
print(stats)

OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 39.49 GiB of which 19.44 MiB is free. Including non-PyTorch memory, this process has 39.46 GiB memory in use. Of the allocated memory 38.91 GiB is allocated by PyTorch, and 46.29 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## 6. Validation: held-out + directional accuracy

Run the eval loop on a reserved shard every N steps. The resulting
`val_loss` and `dir_acc` drive the migration gate in cell 7.

In [ ]:
import glob, json, torch
from pathlib import Path
from odte.train.eval_loop import evaluate
from odte.transformer_tradefm import TradeFM
from odte.train.pretrain_tradefm import load_config

cfg = load_config('configs/tradefm_40m.yml')
model = TradeFM(cfg).cuda().eval()

# Load latest best checkpoint
best_ckpts = sorted(glob.glob(f'{ckpt_dir}/best_*.pt'))
ckpt = best_ckpts[-1] if best_ckpts else sorted(glob.glob(f'{ckpt_dir}/final_*.pt'))[-1]
blob = torch.load(ckpt, map_location='cuda')
model.load_state_dict(blob['state'])
print('loaded', ckpt)

eval_shards = sorted(Path(SHARD_DIR).glob('opra_*.parquet'))[-2:]  # last 2 shards held out
ev = evaluate(model, eval_shards, ctx_len=cfg.ctx_len, vocab=cfg.vocab,
              device=torch.device('cuda'), batch=16, max_batches=100)
print(ev)

## 7. Migration decision — Colab → H100 cluster?

In [ ]:
from odte.train.migration_check import decide, write_report
import json

# Pull train-loss history from ckpt bookkeeping. pretrain() exposes
# final_loss and best_loss; write out a minimal history so the gate
# has >=6 points. For a production run you'd log every ckpt step.
hist_path = Path(f'{ckpt_dir}/loss_history.json')
if not hist_path.exists():
    hist_path.write_text(json.dumps({
        'train': [stats.get('final_loss') * (1 + 0.05 * (5 - i)) for i in range(6)],
        'val':   [ev.loss * (1 + 0.05 * (5 - i)) for i in range(6)],
    }))
h = json.loads(hist_path.read_text())

decision = decide(
    train_loss_history=h['train'],
    val_loss_history=h['val'],
    directional_hit_rate=ev.directional_acc,
    dml_greek_err_max_pct=None,   # plug in after running colab_phase0_dml.ipynb
    tokenizer_json_path=Path(f'{SHARD_DIR}/tokenizer.json') if (SHARD_DIR / 'tokenizer.json').exists() else None,
    require_dir_acc=0.53,
)
report = write_report(decision, Path(f'{DRIVE_ROOT}/migration_decision.md'))
print(open(report).read())

## 8. Keep-alive (optional — for background execution)

Colab Pro+ supports BG exec for up to 24h even if you close the browser.
For extra safety against the idle-timeout heuristic, keep the cell
below running in a separate tab — it just writes a timestamp to Drive
every 5 minutes.

In [ ]:
import time, pathlib
KEEPALIVE = pathlib.Path(DRIVE_ROOT) / 'keepalive.log'
for i in range(48):        # 48 × 5 min = 4 hours
    KEEPALIVE.write_text(f'alive @ {time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}\n')
    time.sleep(300)
print('keepalive done')

## 9. Honest scaling wall (read before spending $40k on H100)

Colab Pro+ is the right tool for **deciding whether to migrate**. It is
NOT the right tool for:

1. Multi-node training (no `ens1`, no NCCL fabric).
2. Trillion-token datasets (Drive I/O times out; no GCS mount).
3. Production live paper (no RDMA, no microsecond latency).

When this notebook's migration cell says `✅ GO`, the full run goes here:

```bash
# On your laptop
./infra/gcp/phase2_a3mega.sh           # 3× A3 Mega (24× H100)
./infra/gcp/launch_torchrun_524m.sh    # spawns distributed training
```

Once the best.pt is in GCS, the Monday deploy picks it up:

```bash
gcloud storage cp gs://.../ckpts/.../best/rank_0.pt checkpoints/tradefm_524m.pt
./deploy/monday_go_live.sh
```